# 📄 Long-Running Job Application Assistant

## What We're Building

A multi-step job application workflow:
1. **Parse JD** — extract key requirements from a job description
2. **Tailor CV** — adjust resume bullet points to match the JD
3. **Write Cover Letter** — personalized cover letter
4. **Store Versions** — save all outputs for later
5. **Review** — human reviews the full package

## Architecture

```
Job Description + Resume
         │
         ▼
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ Parse    │──▶│ Tailor   │──▶│ Cover    │──▶│ Store    │──▶│ ⏸ Review │
│ JD       │   │ CV       │   │ Letter   │   │ Versions │   │ Package  │
└──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
```

## Key LangGraph Concept: Long-Running Multi-Node Pipeline

This project demonstrates a **5-node pipeline** where each step
builds on the prior step's output. The state accumulates a complete
application package by the end.

## Stack
- **LangGraph** — multi-step pipeline with checkpoint review
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)
print("All imports successful!")

## Step 2 — Sample Job Description & Resume

In [ ]:
class ApplicationState(TypedDict):
    job_description: str
    candidate_resume: str
    jd_analysis: str           # Parsed JD requirements
    tailored_cv: str           # Adjusted resume
    cover_letter: str          # Generated cover letter
    application_package: str   # Combined stored version
    review_feedback: str       # Human review notes


sample_jd = """
Senior Machine Learning Engineer — HealthAI Inc. (Remote)

About the Role:
We're building AI-powered diagnostic tools that help doctors detect diseases
earlier. You'll lead ML model development for medical imaging analysis.

Requirements:
- 5+ years of ML/AI engineering experience
- Strong Python skills (PyTorch or TensorFlow)
- Experience with computer vision / medical imaging
- Knowledge of MLOps (MLflow, Kubeflow, or similar)
- Experience deploying models to production at scale
- Healthcare or regulated industry experience is a plus
- MS or PhD in CS, ML, or related field

Nice to have:
- Experience with DICOM/medical image formats
- FDA 510(k) or SaMD regulatory experience
- Publications in medical AI

What we offer:
- $180-220K base + equity
- Full health/dental/vision
- Unlimited PTO, remote-first culture
"""

sample_resume = """
JANE DOE
ML Engineer | jane.doe@email.com | San Francisco, CA

EXPERIENCE:

Senior ML Engineer — TechVision Inc. (2021-Present)
- Built computer vision pipeline for defect detection in manufacturing (PyTorch)
- Deployed 12 ML models to production using MLflow and Kubernetes
- Reduced inference latency by 60% through model quantization and TensorRT
- Led team of 4 engineers, mentored 2 junior engineers
- Implemented CI/CD for ML models with automated testing

ML Engineer — DataCorp (2019-2021)
- Developed NLP models for document classification (BERT fine-tuning)
- Built real-time recommendation system serving 1M+ daily users
- Created data pipelines with Apache Spark and Airflow
- Improved model accuracy by 15% through feature engineering

Junior Data Scientist — StartupXYZ (2017-2019)
- Built predictive models for customer churn (scikit-learn, XGBoost)
- Created dashboards and reports for stakeholders (Tableau)
- Managed data collection and labeling workflows

EDUCATION:
MS Computer Science, Stanford University (2017)
BS Mathematics, UC Berkeley (2015)

SKILLS:
Python, PyTorch, TensorFlow, scikit-learn, MLflow, Kubernetes, Docker,
Apache Spark, SQL, Git, AWS (SageMaker, EC2, S3)
"""

print(f"JD loaded: {len(sample_jd.split())} words")
print(f"Resume loaded: {len(sample_resume.split())} words")

## Step 3 — Define All Pipeline Nodes

In [ ]:
# --- Node 1: Parse Job Description ---
parse_prompt = ChatPromptTemplate.from_template(
    """Analyze this job description and extract structured requirements.

Job Description:
{jd}

Extract:
1. MUST-HAVE skills (required)
2. NICE-TO-HAVE skills (preferred)
3. KEY THEMES (what they really care about)
4. CULTURE SIGNALS (what kind of person they want)
5. RED FLAGS or GOTCHAS (anything unusual in requirements)

Analysis:"""
)


def parse_jd(state: ApplicationState) -> dict:
    print("📋 Node: parse_jd — Extracting requirements...")
    chain = parse_prompt | llm | StrOutputParser()
    analysis = chain.invoke({"jd": state["job_description"]})
    return {"jd_analysis": analysis}


# --- Node 2: Tailor CV ---
tailor_prompt = ChatPromptTemplate.from_template(
    """Tailor this resume for the target job. Rewrite bullet points to
emphasize relevant experience. Keep the same structure but adjust language
to match the JD's requirements.

JD Requirements:
{jd_analysis}

Original Resume:
{resume}

Rules:
- Keep all facts truthful — only reframe, don't fabricate
- Prioritize experiences matching MUST-HAVE skills
- Use keywords from the JD naturally
- Quantify achievements where possible

Tailored Resume:"""
)


def tailor_cv(state: ApplicationState) -> dict:
    print("✏️ Node: tailor_cv — Customizing resume...")
    chain = tailor_prompt | llm | StrOutputParser()
    tailored = chain.invoke({"jd_analysis": state["jd_analysis"], "resume": state["candidate_resume"]})
    return {"tailored_cv": tailored}


# --- Node 3: Write Cover Letter ---
cover_prompt = ChatPromptTemplate.from_template(
    """Write a compelling cover letter for this job application.

Job Requirements:
{jd_analysis}

Candidate Background:
{resume}

Rules:
- 3-4 paragraphs, under 300 words
- Open with a hook (not "I am writing to apply...")
- Connect specific experiences to their requirements
- Show genuine enthusiasm for their mission
- End with a confident but not arrogant CTA
- Professional, warm tone

Cover letter:"""
)


def write_cover_letter(state: ApplicationState) -> dict:
    print("💌 Node: write_cover_letter — Crafting letter...")
    chain = cover_prompt | llm | StrOutputParser()
    letter = chain.invoke({"jd_analysis": state["jd_analysis"], "resume": state["candidate_resume"]})
    return {"cover_letter": letter}


# --- Node 4: Store versions ---
def store_versions(state: ApplicationState) -> dict:
    print("💾 Node: store_versions — Packaging application...")
    package = (
        "=" * 60 + "\n"
        "APPLICATION PACKAGE\n"
        "=" * 60 + "\n\n"
        "--- JD ANALYSIS ---\n"
        f"{state['jd_analysis']}\n\n"
        "--- TAILORED RESUME ---\n"
        f"{state['tailored_cv']}\n\n"
        "--- COVER LETTER ---\n"
        f"{state['cover_letter']}\n\n"
        "=" * 60
    )
    return {"application_package": package}


# --- Node 5: Final review (after human feedback) ---
def compile_final(state: ApplicationState) -> dict:
    """This node just confirms the review is complete."""
    print("✅ Node: compile_final — Review noted, package finalized.")
    return {}  # Review feedback already in state


print("All 5 pipeline nodes defined")

## Step 4 — Build & Run

In [ ]:
workflow = StateGraph(ApplicationState)
workflow.add_node("parse_jd", parse_jd)
workflow.add_node("tailor_cv", tailor_cv)
workflow.add_node("write_cover_letter", write_cover_letter)
workflow.add_node("store_versions", store_versions)
workflow.add_node("compile_final", compile_final)

workflow.set_entry_point("parse_jd")
workflow.add_edge("parse_jd", "tailor_cv")
workflow.add_edge("tailor_cv", "write_cover_letter")
workflow.add_edge("write_cover_letter", "store_versions")
workflow.add_edge("store_versions", "compile_final")
workflow.add_edge("compile_final", END)

memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["compile_final"]  # Pause for human review of full package
)

print("Job application pipeline compiled")

In [ ]:
config = {"configurable": {"thread_id": "job-app-001"}}

print("🚀 Starting job application workflow...\n")
result = app.invoke({
    "job_description": sample_jd,
    "candidate_resume": sample_resume,
    "jd_analysis": "",
    "tailored_cv": "",
    "cover_letter": "",
    "application_package": "",
    "review_feedback": "",
}, config)

print("\n⏸️ PAUSED — Review your application package:")
print("\n" + "="*60)
print("TAILORED RESUME:")
print("="*60)
print(result["tailored_cv"])

print("\n" + "="*60)
print("COVER LETTER:")
print("="*60)
print(result["cover_letter"])

In [ ]:
# Human review and resume
feedback = "Looks great. Minor edit: emphasize the medical/healthcare angle more in the cover letter's opening. Otherwise approved."
app.update_state(config, {"review_feedback": feedback})

final = app.invoke(None, config)
print("\n✅ Application package finalized!")
print(f"Review feedback: {final['review_feedback']}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Multi-node pipeline** | 5 sequential nodes building up state |
| **JD parsing** | Extract structured requirements from freeform text |
| **Tailoring** | Reframe (not fabricate) resume for the role |
| **Package storage** | Combine all outputs into one reviewable package |
| **HITL review** | Human reviews full package before finalizing |

## 🔧 Extensions

- **Multiple versions**: Generate 3 cover letter variations to choose from
- **ATS optimization**: Score resume against common ATS keyword filters
- **Multi-JD batch**: Apply to 10 jobs, tailoring each one
- **Interview prep**: Add a node that generates likely interview questions